# 07. SpQR and SqueezeLLM: Sparse Plus Nonuniform PTQ

- **Category:** 14_quantisation / 03_llm_ptq_algorithms
- **Lineage & Citations:** Dettmers et al. (2023, *SpQR: A Sparse-Quantized Representation for Near-Lossless LLM Weight Compression*); Kim et al. (2023, *SqueezeLLM: Dense-and-Sparse Quantization*).
- **Purpose:** Overcome catastrophic degradation in 3-bit and 2-bit quantization by isolating the top-k most sensitive weights (Hessian diagonal weighted) into a high-precision sparse component, while quantizing remaining dense weights to ultra-low bits.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [06. OmniQuant and HQQ](06_omniquant_and_hqq_optimization_based_ptq.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. Vector Product and Additive Quantization VQ PQ RVQ AQLM](../04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use pure uniform quantization below 4 bits without outlier protection (causes heavy perplexity degradation); do NOT use full dense FP16 if memory bandwidth is the primary bottleneck (isolate $<2\%$ of weights as sparse outliers to maintain $3\times$ throughput gains).

## Mathematical Foundations & Sensitivity-Guided Decomposition

1. **Hessian Sensitivity Metric**:
   $$\mathbf{H}_{ii} = \frac{2}{N} \sum_{k=1}^N \mathbf{X}_{ki}^2, \quad \mathbf{S}_{ij} = \mathbf{H}_{ii} \cdot |W_{ij}|$$
2. **Sparse Outlier Isolation**:
   $$\mathbf{W} = \mathbf{W}_{\text{dense}} + \mathbf{W}_{\text{sparse}}, \quad \mathbf{M}_{ij} = \mathbb{I}[\mathbf{S}_{ij} \ge \tau]$$
3. **Dense 3-bit Quantization**:
   $$\mathbf{q} = \text{clamp}\left(\text{round}\left(\frac{\mathbf{W}_{\text{dense}}}{\Delta}\right), -4, 3\right), \quad \Delta = \frac{\max(|\mathbf{W}_{\text{dense}}|)}{3}$$
4. **SpQR Fused Linear Forward**:
   $$\mathbf{Y} = \mathbf{X} \mathbf{W}_{\text{dense}}^{\text{deq}} + \mathbf{X} \mathbf{W}_{\text{sparse}} + \mathbf{b}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def compute_hessian_sensitivity(x_calib: torch.Tensor, weight: torch.Tensor):
    h_diag = 2.0 * torch.sum(x_calib ** 2, dim=0) / float(x_calib.shape[0])
    return h_diag, h_diag.unsqueeze(1) * torch.abs(weight)

def extract_sparse_outliers(weight: torch.Tensor, sensitivity: torch.Tensor, top_k_count: int = 8):
    threshold = float(torch.topk(sensitivity.flatten(), k=top_k_count).values[-1])
    mask = sensitivity >= threshold
    w_sparse = torch.where(mask, weight, torch.zeros_like(weight))
    w_dense = torch.where(mask, torch.zeros_like(weight), weight)
    return w_sparse, w_dense, mask.float()

def quantize_dense_int3(w_dense: torch.Tensor, mask: torch.Tensor, qmin: float = -4.0, qmax: float = 3.0):
    scale = torch.max(torch.abs(w_dense), dim=0, keepdim=True).values / qmax
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(w_dense / scale), qmin, qmax)
    q = torch.where(mask > 0.0, torch.zeros_like(q), q)
    return q * scale, scale

def spqr_forward(x: torch.Tensor, w_dense_deq: torch.Tensor, w_sparse: torch.Tensor, bias: torch.Tensor | None = None):
    y = x @ w_dense_deq + x @ w_sparse
    return (y + bias) if bias is not None else y

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. Sensitivity and outlier extraction parity
h_diag, sens = compute_hessian_sensitivity(weights["x_calib"], weights["w_orig"])
assert float(torch.max(torch.abs(h_diag - weights["h_diag"]))) < 1e-5
assert float(torch.max(torch.abs(sens - weights["sensitivity"]))) < 1e-5
w_s, w_d, m = extract_sparse_outliers(weights["w_orig"], sens, top_k_count=8)
assert float(torch.max(torch.abs(w_s - weights["w_sparse"]))) < 1e-5
assert float(torch.max(torch.abs(w_d - weights["w_dense"]))) < 1e-5

# 3. Dense 3-bit quantization and SpQR forward parity
w_d_deq, d_scale = quantize_dense_int3(w_d, m, qmin=-4.0, qmax=3.0)
assert float(torch.max(torch.abs(w_d_deq - weights["w_dense_deq"]))) < 1e-5
y_spqr = spqr_forward(weights["x_calib"], w_d_deq, w_s, weights["b_orig"])
assert float(torch.max(torch.abs(y_spqr - weights["out_spqr"]))) < 1e-5
print("All SpQR sensitivity, outlier, and forward assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: Pure 3-bit vs SpQR (3-bit Dense + 6.25% FP16 Sparse Outliers)
y_fp32 = weights["out_fp32"]
sig_pwr = float(torch.mean(y_fp32 ** 2))

def eval_sqnr(y_pred: torch.Tensor) -> float:
    mse = float(torch.mean((y_fp32 - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_pure = eval_sqnr(weights["out_pure_3bit"])
sqnr_spqr = eval_sqnr(weights["out_spqr"])

print(f"Pure 3-bit Dense SQNR: {sqnr_pure:.2f} dB")
print(f"SpQR (Dense+Sparse):   {sqnr_spqr:.2f} dB (+{sqnr_spqr - sqnr_pure:.2f} dB gain!)")
assert sqnr_spqr > sqnr_pure + 1.0, "SpQR must yield >1 dB SQNR gain over pure 3-bit quantization!"
assert sqnr_spqr > 18.0, "SpQR must achieve >18 dB SQNR with 6.25% sparse outliers!"

In [ ]:
# Latency Microbenchmark: Fused Dense + Sparse Projection Forward
b, d_in, d_out = 16, 4096, 4096
x_mock = torch.randn(b, d_in)
wd_mock = torch.randn(d_in, d_out) * 0.1
ws_mock = torch.zeros(d_in, d_out)
ws_mock[:32, :32] = torch.randn(32, 32) * 0.5  # Sparse outliers
bias_mock = torch.randn(d_out) * 0.01

t0 = time.perf_counter()
for _ in range(50):
    weff = wd_mock + ws_mock
    y_mock = x_mock @ weff + bias_mock
t_spqr = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused Dense+Sparse Forward (16x4096x4096) latency: {t_spqr:.3f} ms")
assert t_spqr < 30.0, "SpQR forward projection must complete in <30 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Heavy-Tail Bottleneck**: Below 4-bit, uniform quantization fails because rare sensitive weights expand the grid step size. Isolating $<2\%$ of outlier weights restores accuracy.
2. **Hessian Diagonal Weighting**: Second-order sensitivity $S_{ij} = H_{ii} |W_{ij}|$ accurately isolates critical weights without costly iterative re-inversion.
3. **Fused Sparse Projection**: Expressing weights as $W = W_{\text{dense}} + W_{\text{sparse}}$ enables efficient execution on modern tensor engines.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.md](../../../markdown/14_quantisation/03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.md)
- **PyTorch 2.x Module:** [07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.py](../../../pytorch2/14_quantisation/03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.py)
- **Functional JAX Module:** [07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.py](../../../jax/14_quantisation/03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.py)
- **Native Rust Candle Track:** [07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.rs](../../../candle/14_quantisation/03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.rs)
- **Hardware Kernel (CPU / WGSL):** [07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.rs](../../../kernels/14_quantisation/03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.rs)
- **Golden SafeTensors:** [07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.safetensors](../../../weights/14_quantisation/03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 06. OmniQuant and HQQ](06_omniquant_and_hqq_optimization_based_ptq.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 01. Vector Product and Additive Quantization VQ PQ RVQ AQLM](../04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.ipynb) &rarr;